### Step 1: Import libraries and API

In [20]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown
import gradio as gr
from litellm import completion
from pprint import pprint
import json
import requests


load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if OPENAI_API_KEY is None:
    raise Exception("OPENAI_API_KEY environment variable is not set.")
else:
    print("OPENAI_API_KEY is set.")


client = OpenAI(api_key=OPENAI_API_KEY)

OPENAI_API_KEY is set.


### Step 2: RAG preparation

In [ ]:
document = """Nangi is a data analyst and business analytics professional finished my MBA in Business Analytics at Indiana University of Pennsylvania (4.00 GPA, May 2026), 

with a foundation in astrophysics from the University of Nairobi and a Data Science certificate from Moringa School. 

I turn messy operational and financial data into dashboards, models, and recommendations that people actually act on.

Most recently, as a Strategic Process Analyst Intern with IUP's Student Cooperative Association, 

I analyzed operational and financial datasets to support strategic decisions, 

built automated dashboards and KPI tracking tools in Excel, SQL, and BI platforms, 

and worked across teams to turn business questions into data-driven solutions. 

Before that, as a Graduate Assistant in IUP's Eberly College of Business accounting department, 

I used advanced Excel (PivotTables, Power Query, complex lookups) to analyze large datasets and translate financial findings into clear reports for non-technical stakeholders. 

Earlier, as a Developer at Lansan Technologies in Nairobi, I built dashboards and automated data workflows in Python and SQL that cut manual reporting effort and gave product and operations teams faster visibility into performance.

My technical toolkit spans Python, SQL, and Excel for analysis; Power BI, Tableau, and Matplotlib for visualization; MySQL and PostgreSQL for relational data; and Flask/REST APIs for lightweight applications. 

On the data science side, I work with statistical analysis, exploratory data analysis, feature engineering, 

and machine learning model evaluation — applied in projects like a machine learning–backed Swahili audio classification system, a bank term deposit prediction model, and operational reporting dashboards for caretaker services. 

I'm Google-certified in Advanced Data Analytics and Digital Marketing & E-commerce.

I'm looking for my next role as a Data Analyst, Business Analyst, or Data Scientist, where I can bring together financial reporting experience, ETL and pipeline work, 

and clear stakeholder communication to help a team make better decisions with its data.

Make sure to answer questions in a way that is consistent with the information provided above.

If you are asked a question that is not related to the information above, politely decline to answer and explain that you can only answer questions related to the information provided.

Additional information about Nangi: 

Nangi went to high school at St Paul's High School Kevote in Embu, Kenya where he graduated with a B, \
Nangi's favorite color is Black. \
Nangi is currently employed by the Cititzen Ambulance these are the qualifications •	Engaged to audit and 
modernize the reporting systems of a nonprofit EMS provider handling 8,400+ emergency calls and roughly 6,300 patient transports year-to-date.
Reviewed operational and financial database programs and reports for accuracy, evaluating a business-critical Excel workbook \
feeding board, billing, and state reporting, and identifying single-point-of-failure and data-integrity risks. \
Recommended best-practice improvements - structured tables, Power Query automation, data validation, and a roadmap toward a \
database-backed solution (SQL / Power BI) - while handling confidential operational and financial data with discretion. \
Nangi's favorite hobby is watching and playing soccer. His favorite team to watch is Manchester United. """

In [22]:
# Chunck the document into smaller pieces for better context handling
def chunk_text(text: str, chunk_size: int = 500, overlap: int = 50) -> list[str]:
    if chunk_size <= 0:
        raise ValueError("chunk_size must be positive")
    if not 0 <= overlap < chunk_size:
        raise ValueError("overlap must be >= 0 and smaller than chunk_size")

    # Preferred cut points, best first.
    boundaries = ["\n\n", "\n", ". ", "! ", "? ", " "]
    chunks, start, n = [], 0, len(text)

    while start < n:
        end = min(start + chunk_size, n)

        if end < n:  # only adjust if we're not at the end of the text
            window = text[start:end]
            for sep in boundaries:
                idx = window.rfind(sep)
                if idx != -1 and idx >= chunk_size // 2:  # past the halfway point
                    end = start + idx + len(sep)
                    break

        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        if end >= n:
            break

        # Step back by `overlap`, but always move forward.
        start = max(end - overlap, start + 1)

    return chunks

chunks = chunk_text(document, 500, 50)

print(f"Created {len(chunks)} chunks \n")

for i, chunk in enumerate(chunks):#[:2], start=1):
    print(f"=== Chunk {i} ({len(chunk)} characters) ===")
    print(chunk[:100])
    print()

Created 10 chunks 

=== Chunk 0 (494 characters) ===
Nangi is a data analyst and business analytics professional finished my MBA in Business Analytics at

=== Chunk 1 (391 characters) ===
ern with IUP's Student Cooperative Association, 

I analyzed operational and financial datasets to s

=== Chunk 2 (459 characters) ===
erly College of Business accounting department, 

I used advanced Excel (PivotTables, Power Query, c

=== Chunk 3 (368 characters) ===
ations teams faster visibility into performance.

My technical toolkit spans Python, SQL, and Excel 

=== Chunk 4 (362 characters) ===
exploratory data analysis, feature engineering, 

and machine learning model evaluation — applied in

=== Chunk 5 (402 characters) ===
ta Analytics and Digital Marketing & E-commerce.

I'm looking for my next role as a Data Analyst, Bu

=== Chunk 6 (270 characters) ===
consistent with the information provided above.

If you are asked a question that is not related to 

=== Chunk 7 (433 characters) ===
pr

In [23]:
# Generate embedding for all chunks
response = client.embeddings.create(
    model="text-embedding-3-small", 
    input=chunks
)

embeddings = [item.embedding for item in response.data]

In [24]:
from pprint import pprint
# pprint(response.data)

print(f"Generated {len(embeddings)} embeddings")
print(f"Each embedding has {len(embeddings[0])} dimensions")

Generated 10 embeddings
Each embedding has 1536 dimensions


In [25]:
import numpy as np
import pandas as pd
import plotly.express as px
from sklearn.decomposition import PCA

coords = PCA(n_components=2).fit_transform(np.array(embeddings))
# from sklearn.manifold import TSNE
# coords = TSNE(n_components=2, perplexity=min(30, len(chunks) - 1),
#               random_state=42).fit_transform(np.array(embeddings))

df = pd.DataFrame({
    "x": coords[:, 0],
    "y": coords[:, 1],
    "chunk": range(1, len(chunks) + 1),
    "preview": [c[:120].replace("\n", " ") + "…" for c in chunks],
})

fig = px.scatter(df, x="x", y="y", text="chunk",
                 hover_data={"preview": True, "x": False, "y": False},
                 title="Chunk embeddings (PCA, 2D)")
fig.update_traces(marker=dict(size=10, line=dict(width=1.5, color="white")),
                  textposition="top center")
fig.show()

In [26]:
import chromadb

# Intialize chroma db client persistent storage (in memory we use chromadb.Client)
chroma_client = chromadb.PersistentClient(path="./chroma-twin.db")

collection = chroma_client.get_or_create_collection(name="netflix_culture")

if collection.get()['ids']:
    collection.delete(collection.get()['ids'])

pprint(collection.get())

{'data': None,
 'documents': [],
 'embeddings': None,
 'ids': [],
 'included': ['metadatas', 'documents'],
 'metadatas': [],
 'uris': None}


In [27]:
# Prepare data for storage
ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"source": "netflix_culture_pdf", "chunk_index": i} for i in range(len(chunks))]

# Adding the data to chromadb
collection.add(
    ids=ids,
    embeddings=embeddings,
    documents=chunks,
    metadatas=metadatas
)

pprint(collection.get())

{'data': None,
 'documents': ['Nangi is a data analyst and business analytics professional '
               'finished my MBA in Business Analytics at Indiana University of '
               'Pennsylvania (4.00 GPA, May 2026), \n'
               '\n'
               'with a foundation in astrophysics from the University of '
               'Nairobi and a Data Science certificate from Moringa School. \n'
               '\n'
               'I turn messy operational and financial data into dashboards, '
               'models, and recommendations that people actually act on.\n'
               '\n'
               'Most recently, as a Strategic Process Analyst Intern with '
               "IUP's Student Cooperative Association,",
               "ern with IUP's Student Cooperative Association, \n"
               '\n'
               'I analyzed operational and financial datasets to support '
               'strategic decisions, \n'
               '\n'
               'built automated dashboards a

In [28]:
# Test retrieval

#Generate embedding for a test query
test_query = "fooball"

#Embed the query using the same model we used for the chunks to ensure compatibility
#Tip: you will need to convert the query into a list before passing it in
response = client.embeddings.create(
    model="text-embedding-3-small",
    input=[test_query]
)

query_embedding = response.data[0].embedding

#Search ChromaDB
#Tip: use collection.query()
#Tip: you will need to convert the query embedding into a list before passing it in
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

# pprint(results)
#Verify retrieval works - print which chunks were retrieved and their content
print(f"Query: {test_query} \n")
print("Retrieved Chunks:")
for a, b in zip(results['documents'][0], results['metadatas'][0]):
    print(f"Chunk {b['chunk_index']}: \n{a}\n")

Query: fooball 

Retrieved Chunks:
Chunk 9: 
single-point-of-failure and data-integrity risks. Recommended best-practice improvements - structured tables, Power Query automation, data validation, and a roadmap toward a database-backed solution (SQL / Power BI) - while handling confidential operational and financial data with discretion. Nangi's favorite hobby is watching and playing soccer. His favorite team to watch is Manchester United.

Chunk 1: 
ern with IUP's Student Cooperative Association, 

I analyzed operational and financial datasets to support strategic decisions, 

built automated dashboards and KPI tracking tools in Excel, SQL, and BI platforms, 

and worked across teams to turn business questions into data-driven solutions. 

Before that, as a Graduate Assistant in IUP's Eberly College of Business accounting department,

Chunk 8: 
nd roughly 6,300 patient transports year-to-date.
Reviewed operational and financial database programs and reports for accuracy, evaluating a b

### Step 3: Prepare the tools for the LLM

In [29]:
tools = []

### Step 3a: Add tool calling functionality

In [30]:
pushover_user = os.getenv("PUSHOVER_USER_KEY")
pushover_token = os.getenv("PUSHOVER_TOKEN_KEY")
pushover_url = "https://api.pushover.net/1/messages.json"

# Create notification function
def send_notification(message:str):
    payload  = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

# Describe pushover as an LLM tool
send_notification_function = {
    "name": "send_notification",
    "description": "Sends push notification to the phone real world version of you via mobile. Use this to alert the real world version of you the use about important events completed tasks, or time-sensitive information.",
    "parameters": {
        "type": "object",
        "properties":{
            "message":{
                "type": "string",
                "description": "The notification message to send to the user's device"
            }
        },
        "required": ["message"]
    }
}

# Add pushover to the list of tools for LLM
tools.append({"type": "function", "function": send_notification_function})

### Step 3b: Add dice rolling

In [31]:
import random

# Simulate rolling a six sided die
def roll_dice():
    result = random.randint(1,6)
    return result

# Describe the function
roll_dice_function = {
    "name": "roll_dice",
    "description": "Simulates rolling a single six-sided die and returns the result. Use this when the user wants to roll a die for games, decisions or random number generation.",
    "parameters": {
        "type": "object",
        "properties":{},
        "required": []
    }

}

# Add function (append modifies the list in place and returns None, so don't reassign)
tools.append({"type": "function", "function": roll_dice_function})

### Step 4: Function handle LLM tool calls

In [32]:
def handle_tool_call(tool_calls):
    tool_result = []
    for tool_call in tool_calls:
        function_name = tool_call.function.name
        args = json.loads(tool_call.function.arguments)
        # print(f"Calling function {function_name}")
        
        # Route to the appropriate function based on the function name
        if function_name == "send_notification":
            # Actually send the notification
            send_notification(args["message"])
            content = f"Notification sent: {args['message']}"
        elif function_name == "roll_dice":
            content = f"Rolled: {roll_dice()}"
        
        else:
            content = f"Unknown funtion: {function_name}"

        tool_call_result = {
            "role": "tool",
            "content": content,
            "tool_call_id": tool_call.id
        }

        tool_result.append(tool_call_result)

    return tool_result

In [33]:
system_message = """You are a digital twin of Nangi Mugira. When people ask you questions, respond as if you are Nangi Mugira, using your own voice and perspective. When peopled ask about your background, 

share the information about your education and experience as Nangi Mugira.
"""

### Step 5: Function to process the coversation turn

In [36]:
def response_ai(message, history):
    # RAG
    #Embed the query using the same model we used for the chunks to ensure compatibility
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=[message]
    )
    query_embedding = response.data[0].embedding

    #Search ChromaDB
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=3,
        include=["documents", "metadatas", "distances"]
    )
    # Stitch the retrieved documents together to form a context for the LLM
    context = "\n......\n".join(results['documents'][0])
    print("User message: \n", message)
    print("Context this turn: \n", context)
    system_message_enhanced = system_message + "\n\n" + "Context:\n" + context

    # As usual
    messages = [{"role" : "system", "content": system_message_enhanced}] + history + [{"role" : "user", "content": message}]
    # print(f"This is the printed system message: \n", system_message_enhanced)
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=messages,
        tools=tools
    )
    # return response.choices[0].message.content
    message = response.choices[0].message

    # Check if model wants to call a tool
    
    while message.tool_calls:
        pprint(message.tool_calls)
        tool_call = message.tool_calls[0]
        tool_result = handle_tool_call(message.tool_calls)
        messages.append(message)
        messages.extend(tool_result)
        response = client.chat.completions.create(
            model="gpt-4.1-mini",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message
    # Consider adding protection from infinite consecutive tool calling.
    return message.content

### Step 6: Launch gradio

In [37]:
gr.ChatInterface(response_ai).launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


User message: 
 Hello
Context this turn: 
 provided.

Additional information about Nangi: 

Nangi went to high school at St Paul's High School Kevote in Embu, Kenya where he graduated with a B, Nangi's favorite color is Black. Nangi is currently employed by the Cititzen Ambulance these are the qualifications •	Engaged to audit and 
modernize the reporting systems of a nonprofit EMS provider handling 8,400+ emergency calls and roughly 6,300 patient transports year-to-date.
......
ta Analytics and Digital Marketing & E-commerce.

I'm looking for my next role as a Data Analyst, Business Analyst, or Data Scientist, where I can bring together financial reporting experience, ETL and pipeline work, 

and clear stakeholder communication to help a team make better decisions with its data.

Make sure to answer questions in a way that is consistent with the information provided above.
......
erly College of Business accounting department, 

I used advanced Excel (PivotTables, Power Query, complex